# prompts

In [1]:
"""
prompts class 계층도

BasePromptTemplate --> PipelinePromptTemplate
                       StringPromptTemplate --> PromptTemplate
                                                FewShotPromptTemplate
                                                FewShotPromptWithTemplates
                       BaseChatPromptTemplate --> AutoGPTPrompt
                                                  ChatPromptTemplate --> AgentScratchPadChatPromptTemplate



BaseMessagePromptTemplate --> MessagesPlaceholder
                              BaseStringMessagePromptTemplate --> ChatMessagePromptTemplate
                                                                  HumanMessagePromptTemplate
                                                                  AIMessagePromptTemplate
                                                                  SystemMessagePromptTemplate
"""
None

# API Key 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
print(f'{os.environ['OPENAI_API_KEY'][:20]}...')

sk-proj-iKU13YeoxNgF...


# 기본 import

In [4]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [5]:
chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

# PromptTemplate

In [6]:
# 방식1
t = PromptTemplate.from_template("What is the capital of {country}")
t.format(country='France')

'What is the capital of France'

In [7]:
# 방식2
t2 = PromptTemplate(
    template="What is the capital of {country}",
    input_variables=['country'],   # 입력변수들을 PromptTemplate 에 알려주어야 한다.
)

t2.format(country='France')

'What is the capital of France'

# FewShotPromptTemplate
모델에 예제(example) 주기

In [8]:
# 모델에게 '어떻게 대답해야 하는 지에 대한 예제(example)'를 AI 모델에게 주는 것이
# prompt 를 사용해서 '어떻게 대답해야 하는지 알려주는 것'보다 훨씬 좋다

# FewShotPromptTemplate 이 하는 일이 바로 그거다!
# - 이를 통해 예제(샘플)를 형식화(포맷) 할수 있다.
# - 이런 예제들을 데이터베이스등에 저장시켜놓고 활용할수도 있다

# 예시들을 활용하는 예
# 고객지원 봇.
#   다른 고객들ㅇ과의 대화 기록, 많은 기록들...
#   LLM 에게 '어떻게 고객에게 대응' 하는지 알려줄때.

In [9]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [10]:
#예제들 (examples) 준비
# 모델더러 나에게 '이런 식ㅇ로 답벼해 줬으면 좋겟다' 라고 제시.

examples = [
    {
       "question": "What do you know about France?",

        # 원하는 형식의 답변..
        "answer": """
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro        
        """,
    },
  {
    "question": "What do you know about Italy?",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Greece?",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },
    
]


In [11]:
# 일단 예제 없이 호출하면?
chat.invoke("What do you know about France?")

France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower and the Louvre Museum. France is also known for its wine production, fashion industry, and art scene. The country has a diverse landscape, including mountains, beaches, and countryside. French is the official language, and the currency is the Euro. France is a member of the European Union and is one of the most visited countries in the world.

AIMessage(content='France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower and the Louvre Museum. France is also known for its wine production, fashion industry, and art scene. The country has a diverse landscape, including mountains, beaches, and countryside. French is the official language, and the currency is the Euro. France is a member of the European Union and is one of the most visited countries in the world.', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dbea6-9a1f-7170-8451-c569accfbf1c', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 110, 'total_tokens': 124, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
# FewShotPromptTemplate 사용

# 포맷 준비 <- exmples 로 포맷팅할거다
example_template = """
    Human: {question}
    AI: {answer}
"""
# ↑ {question} 과 {answer} 는 위 examples 와 동일한 key 를 사용하여 작성해두어야 한다.


In [13]:
example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n')

In [14]:
print(example_prompt.format(**examples[2]))
# print(example_prompt.format(
#     answer = examples[2]['answer'],
#     question = examples[2]['question']
# ))


    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      



In [15]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,  # 사용할 prompt
    examples=examples,   # 준비된 예제들

    # 유저의 질문을 넣어주자
    suffix="Human: What do you know abount {country}?",   # 질문은 example 과 동일한 형식
    input_variables=['country']
)

prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro        \n        '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know abount {country}?')

In [16]:
print(prompt.format(country='Germany'))


    Human: What do you know about France?
    AI: 
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro        
        



    Human: What do you know about Italy?
    AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      



    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      


Human: What do you know abount Germany?


In [17]:
# step1  example 리스트를 만들고   examples
# step2  FewShotPromptTemplate 에 전달했고 examples=
# step3  어떻게 전달한 예제들을 형식화 할지 알려주었고
# step4  마지막에 질문을 포함시켰다.  suffix, input_variables

# AI 는 우리의 예제들과 똑같은 구조, 형태로 답변하게 될겁니다


In [18]:
chain = prompt | chat

chain

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro        \n        '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know abount {country}?')
| ChatOpenAI(callbacks=[<langchain_core.callbacks.streaming_stdout.Streaming

In [19]:
chain.invoke({
    "country": "Thailand",
})

AI: 
      I know this:
      Capital: Bangkok
      Language: Thai
      Food: Pad Thai and Tom Yum Goong
      Currency: Thai Baht

AIMessage(content='AI: \n      I know this:\n      Capital: Bangkok\n      Language: Thai\n      Food: Pad Thai and Tom Yum Goong\n      Currency: Thai Baht', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dbea6-a687-7342-b7cb-6f990ff079d4', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 36, 'total_tokens': 189, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [20]:
chain.invoke({
    "country": "Turkey",
})

AI: 
      I know this:
      Capital: Ankara
      Language: Turkish
      Food: Kebab and Baklava
      Currency: Turkish Lira

AIMessage(content='AI: \n      I know this:\n      Capital: Ankara\n      Language: Turkish\n      Food: Kebab and Baklava\n      Currency: Turkish Lira', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dbea6-ae45-7ac2-9895-716384df3ea5', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 35, 'total_tokens': 188, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# FewShotChatMessagePromptTemplate

In [21]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

In [22]:
examples = [
    {
       "country": "France",
        "answer": """
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro        
        """,
    },
  {
    "country": "Italy",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "country": "Greece",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },
    
]

In [23]:
example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'What do you know abount {country}?'),  # 반드시 example 과 동일한 key 로!
    ('ai', '{answer}')
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples=examples,
    # suffix= 는 필요없다.
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers."),
    example_prompt,  # !!
    ('human', 'What do you know about {country}?'),  # example_prompt 의 human 메세지와 동일한 포맷
            # 이래서 suffix= 는 필요 없던 것이다.
])


In [24]:
final_prompt.format_messages(country='Germany')

[SystemMessage(content='You are a geography expert, you give short answers.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What do you know abount France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro        \n        ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Italy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Greece?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Athens\n      L

In [25]:
chain = final_prompt | chat

chain.invoke({"country": "Germany"})


      I know this:
      Capital: Berlin
      Language: German
      Food: Bratwurst and Sauerkraut
      Currency: Euro

AIMessage(content='\n      I know this:\n      Capital: Berlin\n      Language: German\n      Food: Bratwurst and Sauerkraut\n      Currency: Euro', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dbea6-b2b1-7ab0-a3c2-8d1f4c3ff2ea', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 33, 'total_tokens': 203, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
chain.invoke({"country": "South Korea"})


      I know this:
      Capital: Seoul
      Language: Korean
      Food: Kimchi and Bibimbap
      Currency: South Korean Won

AIMessage(content='\n      I know this:\n      Capital: Seoul\n      Language: Korean\n      Food: Kimchi and Bibimbap\n      Currency: South Korean Won', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019dbea6-b6ba-7242-89c9-e479dcf2343e', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 32, 'total_tokens': 203, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# ExampleSelector

## LengthBasedExampleSelector

In [27]:
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [ ]:
# LengthBasedExampleSelector 는 기본적으로
# - 예제(example) 들을 형식화 할 수 있고
# - 예제의 양이 얼마나 되는지를 확인할수 있다.

# 그러면, 사용자가 설정해 놓은 세팅값에 따라 prompt 에 알맞은 예제를 골라준다.

In [28]:
examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

In [29]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='Human: {country}\nAI: {answer}')

In [30]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,  # 포맷팅되는 양을 계산하기 위해선 example_prompt
    max_length=10,   # 예제의 양을 얼마나 허용해줄지 지정.  max_length 값보타 긴 예제는 제외.
                #  LengthBasedExampleSelector 의 max_length 는 '단어의 개수' 단위
                #  '글자의 개수' 나 '토큰의 개수'가 아니다
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    # examples=  대신에 example_selector= 지정
    # examples= <- '모든' 예제를 가지고 포맷팅.
    example_selector=example_selector,

    suffix="Human: What do you know about {country}?",
    input_variables=['country']
)

# 포맷팅 동작 확인
prompt.format(country="Brazil")


'Human: What do you know about Brazil?'

In [ ]:
# ↑ 포맷팅된 예제가 하나도 없다!?
# 이유: max_length= 값이 너 무 작다!

In [32]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt, 
    max_length=80,  # max_length 값을 늘려보자.
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,

    suffix="Human: What do you know about {country}?",
    input_variables=['country']
)

# 포맷팅 동작 확인
print(prompt.format(country="Brazil"))

Human: France
AI: 
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        

Human: What do you know about Brazil?


In [33]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt, 
    max_length=180,  # max_length 값을 늘려보자.
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,

    suffix="Human: What do you know about {country}?",
    input_variables=['country']
)

# 포맷팅 동작 확인
print(prompt.format(country="Brazil"))

Human: France
AI: 
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        

Human: Italy
AI: 
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        

Human: What do you know about Brazil?


## Custom ExampleSelector (BaseExampleSelector)

In [34]:
from langchain_core.example_selectors.base import BaseExampleSelector

In [35]:
# BaseExampleSelector 의 구현객체를 만드려면
#  상속 받은뒤 select_examples() 과 add_example() 을 반드시 오버라이딩 해주어야 한다.

class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples):
        self.examples = examples  # 이번예제에선 examples 는 list

    # select_examples() 
    # 입력에 따라 어떠한 샘플을 사용할지 select 함
    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]  # examples 에서 무작위 1개 선택하여 list로 만들어 리턴.
    
    # add_example() 
    # 이미 존재하는 example 에 example 을 추가
    def add_example(self, example):
        self.examples.append(example)




In [52]:
example_selector = RandomExampleSelector(examples=examples)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,

    suffix="Human: What do you know about {country}?",
    input_variables=['country']
)

# 포맷팅 동작 확인
print(prompt.format(country="Brazil"))

Human: Italy
AI: 
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        

Human: What do you know about Brazil?


# PromptTemplate 저장 / 읽어오기
- load_prompt()
- prompt 를 'JSON' 혹은 'YAML' 파일로 저장할수 있다.

## json 파일

In [53]:
# json 파일 생성
with open('prompt.json', 'w') as f:
    f.write("""
  {
    "_type":"prompt",
    "template":"What is the capital of {country}",
    "input_variables":["country"]
  }    
    """)

In [54]:
!type prompt.json


  {
    "_type":"prompt",
    "template":"What is the capital of {country}",
    "input_variables":["country"]
  }    
    


In [55]:
from langchain_core.prompts.loading import load_prompt

In [56]:
# JSON 파일 -> PromptTemplate 객체
prompt = load_prompt('./prompt.json')

prompt

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_22164\4287375961.py:2: LangChainDeprecationWarning: The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0. Use `Use `dumpd`/`dumps` from `langchain_core.load` to serialize prompts and `load`/`loads` to deserialize them.` instead.
  prompt = load_prompt('./prompt.json')


PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}')

In [59]:
prompt.format(country='Germany')

'What is the capital of Germany'

## yaml 파일

In [ ]:
# name: value
#  ★ name 은 쌍따옴표 없다.  : 다음에 한칸 꼭 띄우기!   뒤에 콤마 없다


In [60]:
with open('prompt.yaml', 'w') as f:
    f.write("""
_type: "prompt"
template: "What is the capital of {country}"
input_variables: ["country"]    
    """)

In [61]:
!type prompt.yaml


_type: "prompt"
template: "What is the capital of {country}"
input_variables: ["country"]    
    


In [62]:
prompt = load_prompt('prompt.yaml')

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}')

In [63]:
prompt.format(country='Korea')

'What is the capital of Korea'

# Caching

In [ ]:
# Caching 을 사용하면 모델의 응답을 저장할수 있다.
# 예를들어
# 똑같은 질문을 받는 상황이라면 그 때마다 답변생성할 필요 없이
# 이미 캐싱된 답변을 재사용 할수 있는 것이다 -->  비용절감! 

## set_llm_cache(), InMemoryCache

In [64]:
from langchain_core.globals import set_llm_cache

In [65]:
from langchain_core.caches import InMemoryCache

In [66]:
# ↓ 이와 같이 세팅하면 LLM 의 모든 response 가 '메모리'에 캐시 된다.
set_llm_cache(InMemoryCache())

In [67]:
chat = ChatOpenAI(
    temperature=0.1,
)

In [68]:
# 시간 측정하는 함수
import time
from datetime import timedelta

def check_laptime(message):
    start_time = time.time()
    response = chat.invoke(message)
    end_time = time.time()
    elapsed_time = end_time - start_time  # 경과시간
    print('▶ 경과시간 %s' % (str(timedelta(seconds = elapsed_time))))
    print(f'{len(response.content)} 글자: {response.content}\n')    



In [69]:
check_laptime('How do you make Italian pizza')

▶ 경과시간 0:00:05.261581
1579 글자: To make Italian pizza, you will need the following ingredients:

- 2 1/4 cups of all-purpose flour
- 1 teaspoon of salt
- 1 teaspoon of sugar
- 1 packet of active dry yeast
- 1 cup of warm water
- 2 tablespoons of olive oil
- Tomato sauce
- Mozzarella cheese
- Toppings of your choice (such as pepperoni, mushrooms, bell peppers, etc.)

Here is a step-by-step guide to making Italian pizza:

1. In a large mixing bowl, combine the flour, salt, and sugar. In a separate small bowl, dissolve the yeast in warm water and let it sit for about 5 minutes until it becomes frothy.

2. Pour the yeast mixture and olive oil into the flour mixture and stir until a dough forms. Knead the dough on a floured surface for about 5-7 minutes until it becomes smooth and elastic.

3. Place the dough in a greased bowl, cover it with a clean kitchen towel, and let it rise in a warm place for about 1-2 hours until it doubles in size.

4. Preheat your oven to 475°F (245°C) and place a 

In [70]:
# 동일한 프롬프트로 두번째 호출시, 
# 첫번째 호출로 cache 된 결과 리턴.  (llm 호출 발생 안함.)
check_laptime('How do you make Italian pizza')

▶ 경과시간 0:00:00.003988
1579 글자: To make Italian pizza, you will need the following ingredients:

- 2 1/4 cups of all-purpose flour
- 1 teaspoon of salt
- 1 teaspoon of sugar
- 1 packet of active dry yeast
- 1 cup of warm water
- 2 tablespoons of olive oil
- Tomato sauce
- Mozzarella cheese
- Toppings of your choice (such as pepperoni, mushrooms, bell peppers, etc.)

Here is a step-by-step guide to making Italian pizza:

1. In a large mixing bowl, combine the flour, salt, and sugar. In a separate small bowl, dissolve the yeast in warm water and let it sit for about 5 minutes until it becomes frothy.

2. Pour the yeast mixture and olive oil into the flour mixture and stir until a dough forms. Knead the dough on a floured surface for about 5-7 minutes until it becomes smooth and elastic.

3. Place the dough in a greased bowl, cover it with a clean kitchen towel, and let it rise in a warm place for about 1-2 hours until it doubles in size.

4. Preheat your oven to 475°F (245°C) and place a 

## set_debug()

In [71]:
from langchain_core.globals import set_debug

In [72]:
set_debug(True)

In [73]:
chat.invoke('How do you make Italian pizza')

[llm/start] [llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: How do you make Italian pizza"
  ]
}
[llm/end] [llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "To make Italian pizza, you will need the following ingredients:\n\n- 2 1/4 cups of all-purpose flour\n- 1 teaspoon of salt\n- 1 teaspoon of sugar\n- 1 packet of active dry yeast\n- 1 cup of warm water\n- 2 tablespoons of olive oil\n- Tomato sauce\n- Mozzarella cheese\n- Toppings of your choice (such as pepperoni, mushrooms, bell peppers, etc.)\n\nHere is a step-by-step guide to making Italian pizza:\n\n1. In a large mixing bowl, combine the flour, salt, and sugar. In a separate small bowl, dissolve the yeast in warm water and let it sit for about 5 minutes until it becomes frothy.\n\n2. Pour the yeast mixture and olive oil into the flour mixture and stir until a dough forms. Knead the dough on a floured surface for about 5-7 minutes until it becomes smoot

AIMessage(content='To make Italian pizza, you will need the following ingredients:\n\n- 2 1/4 cups of all-purpose flour\n- 1 teaspoon of salt\n- 1 teaspoon of sugar\n- 1 packet of active dry yeast\n- 1 cup of warm water\n- 2 tablespoons of olive oil\n- Tomato sauce\n- Mozzarella cheese\n- Toppings of your choice (such as pepperoni, mushrooms, bell peppers, etc.)\n\nHere is a step-by-step guide to making Italian pizza:\n\n1. In a large mixing bowl, combine the flour, salt, and sugar. In a separate small bowl, dissolve the yeast in warm water and let it sit for about 5 minutes until it becomes frothy.\n\n2. Pour the yeast mixture and olive oil into the flour mixture and stir until a dough forms. Knead the dough on a floured surface for about 5-7 minutes until it becomes smooth and elastic.\n\n3. Place the dough in a greased bowl, cover it with a clean kitchen towel, and let it rise in a warm place for about 1-2 hours until it doubles in size.\n\n4. Preheat your oven to 475°F (245°C) and 

In [74]:
set_debug(False)

## SQLiteCache

In [75]:
# 데이터베이스에 캐시 하기

In [77]:
from langchain_community.cache import SQLiteCache

In [78]:
set_llm_cache(SQLiteCache('cache.db'))  # 생성될 데이터베이스 파일명

In [79]:
! dir

 D 드라이브의 볼륨: 새 볼륨
 볼륨 일련 번호: 02B1-19A2

 D:\MCP2601\AgentWork\01_LangChain기초 디렉터리

2026-04-24  오후 08:11    <DIR>          .
2026-04-24  오후 08:11    <DIR>          ..
2026-04-23  오후 08:46    <DIR>          .ipynb_checkpoints
2026-04-23  오후 08:44            87,898 01 Hello LangChain.ipynb
2026-04-24  오후 08:09            61,379 02 ModelIO.ipynb
2026-04-24  오후 08:11            32,768 cache.db
2026-04-24  오후 07:34               130 prompt.json
2026-04-24  오후 07:41               103 prompt.yaml
               5개 파일             182,278 바이트
               3개 디렉터리  458,494,078,976 바이트 남음


In [80]:
check_laptime("How do you make Italian pasta")

▶ 경과시간 0:00:03.270763
910 글자: To make Italian pasta, you will need the following ingredients:

- 2 cups of all-purpose flour
- 2 large eggs
- Pinch of salt

Here is a simple step-by-step guide to making Italian pasta:

1. On a clean work surface, pour the flour and make a well in the center.
2. Crack the eggs into the well and add a pinch of salt.
3. Using a fork, gradually mix the eggs into the flour until a dough forms.
4. Knead the dough for about 10 minutes until it becomes smooth and elastic.
5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.
6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.
7. Cut the dough into your desired shape, such as fettuccine or spaghetti.
8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes or until al dente.
9. Drain the pasta and toss with your favorite sauce or toppings.

Enjoy your homemade Italian pasta!



In [81]:
check_laptime("How do you make Italian pasta")

▶ 경과시간 0:00:00.003989
910 글자: To make Italian pasta, you will need the following ingredients:

- 2 cups of all-purpose flour
- 2 large eggs
- Pinch of salt

Here is a simple step-by-step guide to making Italian pasta:

1. On a clean work surface, pour the flour and make a well in the center.
2. Crack the eggs into the well and add a pinch of salt.
3. Using a fork, gradually mix the eggs into the flour until a dough forms.
4. Knead the dough for about 10 minutes until it becomes smooth and elastic.
5. Wrap the dough in plastic wrap and let it rest for at least 30 minutes.
6. After resting, roll out the dough using a pasta machine or a rolling pin until it is thin.
7. Cut the dough into your desired shape, such as fettuccine or spaghetti.
8. Cook the pasta in a large pot of boiling salted water for 2-3 minutes or until al dente.
9. Drain the pasta and toss with your favorite sauce or toppings.

Enjoy your homemade Italian pasta!



In [82]:
set_debug(False)
set_llm_cache(None)  # 캐시 사용 안함

# OpenAI 모델 호출 비용 확인

In [83]:
from langchain_community.callbacks.manager import get_openai_callback

In [84]:
with get_openai_callback() as usage:
    # 이 with 안에서 LLM 호출하면 비용계산이 usage 에 담긴다.
    chat.invoke("How do you make Italian pasta")
    print('🟦', usage)

🟦 Tokens Used: 235
	Prompt Tokens: 13
		Prompt Tokens Cached: 0
	Completion Tokens: 222
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0003395


In [85]:
with get_openai_callback() as usage:    
    a = chat.invoke("How do you make soju")
    b = chat.invoke("How do you make Italian pasta")
    print('\n😀', a.content)
    print('\n😀', b.content)
    print('\n🟦', usage)


😀 Soju is a Korean alcoholic beverage that is typically made from rice, barley, or wheat. Here is a basic recipe for making soju at home:

Ingredients:
- 2 cups of rice
- 4 cups of water
- 1 cup of nuruk (fermentation starter)
- 1 cup of sugar

Instructions:
1. Rinse the rice thoroughly and soak it in water for at least 1 hour.
2. Drain the rice and steam it until it is fully cooked.
3. Allow the cooked rice to cool to room temperature.
4. In a large container, mix the nuruk with the cooked rice and water.
5. Cover the container with a clean cloth and let it ferment for 3-4 days in a warm, dark place.
6. After fermentation, strain the mixture through a cheesecloth to remove any solids.
7. Add sugar to the strained liquid and mix well until dissolved.
8. Transfer the liquid to a clean bottle and let it age for at least 1 week before serving.

Please note that making soju at home can be a lengthy process and may require some trial and error to achieve the desired flavor and alcohol cont

In [86]:
usage.total_cost

0.000745

In [87]:
usage.total_tokens

514

# 모델 config 를 저장하고 불러오기

In [88]:
llm = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model='gpt-4-turbo',
)

# 모델 config 저장
llm.save('model.json')

In [89]:
!type model.json

{
    "model_name": "gpt-4-turbo",
    "temperature": 0.1,
    "top_p": 1,
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "n": 1,
    "seed": null,
    "logprobs": null,
    "max_tokens": 450,
    "_type": "openai"
}


In [ ]:
"""
    ↓답변의 창의성 / 무작위성을 조정하는 핵심 파라미터 
    
    "temperature": 0.1,  : 확률의 분포를 조정.
    "top_p": 1,          : 후보군의 범위(개수) 조정.
"""
None


In [90]:
from langchain_community.llms.loading import load_llm

In [91]:
# 저장된 모델 config 읽어오기
chat = load_llm('model.json')

D:\MCP2601\AgentWork\.venv\Lib\site-packages\langchain_community\llms\openai.py:255: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
D:\MCP2601\AgentWork\.venv\Lib\site-packages\langchain_community\llms\openai.py:1089: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(


In [92]:
chat

OpenAIChat(client=APIRemovedInV1Proxy, model_kwargs={'model_name': 'gpt-4-turbo', 'temperature': 0.1, 'top_p': 1, 'frequency_penalty': 0, 'presence_penalty': 0, 'n': 1, 'seed': None, 'logprobs': None, 'max_tokens': 450}, prefix_messages=[])